# SA Solar — Cape Town Rooftop Solar Detection

This notebook runs the SA_Solar detection & evaluation pipeline on Google Colab.

**Prerequisites:**
- Colab GPU runtime (Runtime → Change runtime type → T4 GPU)
- Large files (tiles, checkpoints) stored in Google Drive under `MyDrive/SA_Solar_Data/`

**Google Drive layout:**
```
MyDrive/SA_Solar_Data/
├── tiles/
│   └── G1238/          # GeoTIFF tiles
├── checkpoints/        # Model weights (.pth)
└── results/            # Detection outputs (auto-created)
```

## 1. Clone Repo & Install Dependencies

In [ ]:
import os

# Clone repo (skip if already cloned)
REPO_DIR = "/content/SA_Solar"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Robertgao0818/SA_Solar.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!bash scripts/colab_setup.sh

## 2. Mount Google Drive & Configure Paths

This creates symlinks so scripts find `tiles/`, `checkpoints/`, `results/` on Drive.

In [ ]:
import sys
sys.path.insert(0, "/content/SA_Solar")

from scripts.colab_config import setup_colab

paths = setup_colab(drive_data_dir="SA_Solar_Data")
print("Configured paths:")
for k, v in paths.items():
    print(f"  {k}: {v}")

## 3. Verify Environment

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training requires CUDA.")
    print("Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Check data availability
from pathlib import Path

project = Path("/content/SA_Solar")
grid_id = "G1238"

tiles_dir = project / "tiles" / grid_id
gt_file = project / "data" / "annotations" / f"{grid_id}_SAM2_260320.gpkg"
checkpoint = project / "checkpoints" / "v2_sam2_260320" / "best_model.pth"

print(f"Tiles dir exists: {tiles_dir.exists()}")
if tiles_dir.exists():
    tifs = list(tiles_dir.glob("*.tif"))
    print(f"  Tile count: {len(tifs)}")
print(f"Ground truth exists: {gt_file.exists()}")
print(f"Checkpoint exists: {checkpoint.exists()}")

## 4. Download Tiles (if needed)

If tiles are not on Drive yet, download them via WMS.

In [ ]:
# Download tiles for a grid (skip if already on Drive)
GRID_ID = "G1238"

tiles_dir = Path(f"tiles/{GRID_ID}")
if not tiles_dir.exists() or len(list(tiles_dir.glob("*.tif"))) == 0:
    print(f"Downloading tiles for {GRID_ID}...")
    !python scripts/imagery/download_tiles.py --grid-id {GRID_ID}
else:
    print(f"Tiles already present: {len(list(tiles_dir.glob('*.tif')))} files")

## 5. Run Detection & Evaluation

Run the full inference + evaluation pipeline.

In [ ]:
# Stock model (geoai pretrained)
!python detect_and_evaluate.py --grid-id G1238

In [ ]:
# Fine-tuned model (if checkpoint exists)
!python detect_and_evaluate.py \
    --grid-id G1238 \
    --model-path checkpoints/v2_sam2_260320/best_model.pth \
    --force

## 6. View Results

In [ ]:
import pandas as pd
from IPython.display import display, Image

results_dir = Path(f"results/{GRID_ID}")

# Presence metrics
metrics_file = results_dir / "presence_metrics.csv"
if metrics_file.exists():
    print("=== Presence Metrics (P/R/F1) ===")
    display(pd.read_csv(metrics_file))

# Footprint metrics
fp_file = results_dir / "footprint_metrics.csv"
if fp_file.exists():
    print("\n=== Footprint IoU Metrics ===")
    display(pd.read_csv(fp_file))

In [ ]:
# Visualizations
for png_name in ["confidence_histogram.png", "precision_recall_curve.png", "iou_threshold_metrics.png"]:
    png_path = results_dir / png_name
    if png_path.exists():
        print(f"\n{png_name}")
        display(Image(filename=str(png_path), width=600))

## 7. Fine-Tuning (Optional)

Export COCO dataset and train a fine-tuned model.

In [ ]:
# Step 1: Export COCO dataset
!python export_coco_dataset.py --output-dir data/coco

In [ ]:
# Step 2: Train (requires GPU)
!python train.py --coco-dir data/coco --output-dir checkpoints/colab_run

In [ ]:
# Step 3: Evaluate with new checkpoint
!python detect_and_evaluate.py \
    --grid-id G1238 \
    --model-path checkpoints/colab_run/best_model.pth \
    --force

## 8. Save Results to Drive

Results are automatically saved to Drive via the symlinks set up in step 2.
You can also manually copy specific files:

In [ ]:
# Verify results are on Drive
drive_results = Path("/content/drive/MyDrive/SA_Solar_Data/results")
if drive_results.exists():
    for f in sorted(drive_results.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(drive_results)}")
else:
    print("Drive results directory not found — results are in local /content/SA_Solar/results/")